# Failure analysis: class confusion and condition robustness

Loads the trained checkpoint (`src/best_model.pth`) and answers two questions
that raw accuracy/F1 numbers don't:

1. **Which sign classes get confused with each other**, and what do those
   mix-ups actually look like?
2. **How does accuracy degrade** under simulated glare, motion blur, and low
   light — the three dashcam conditions the training augmentation was meant
   to approximate?

Run from the `notebooks/` directory (locally or Colab); it adds the repo
root to `sys.path` so `src.*` imports work without installing the package.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import GTSRB
from sklearn.metrics import confusion_matrix

from src.config import (
    DATA_DIR, DEVICE, IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD,
    NUM_CLASSES, NUM_WORKERS, OUTPUT_DIR, ROOT_DIR, SEED,
)
from src.labels import CLASS_NAMES
from src.model import build_model
from src.perturbations import PERTURBATIONS
from src.utils import load_checkpoint

print("Device:", DEVICE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load model and clean test set

Evaluated on the held-out GTSRB **test** split (not validation), matching
`src/evaluate.py`.

In [ ]:
WEIGHTS_PATH = ROOT_DIR / "src" / "best_model.pth"

model = build_model(NUM_CLASSES).to(DEVICE)
checkpoint = load_checkpoint(model, WEIGHTS_PATH, device=DEVICE)
model.eval()

print(f"Loaded {WEIGHTS_PATH}", end="")
if "val_acc" in checkpoint:
    print(f" (epoch={checkpoint.get('epoch')}, val_acc={checkpoint['val_acc']:.4f})")
else:
    print()


In [ ]:
def eval_transform(severity=0.0, perturbation=None):
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    if perturbation is not None and severity > 0:
        ops.append(PERTURBATIONS[perturbation](severity))
    ops += [transforms.ToTensor(), transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]
    return transforms.Compose(ops)


clean_test_set = GTSRB(root=str(DATA_DIR), split="test", transform=eval_transform(), download=True)
clean_test_loader = DataLoader(
    clean_test_set, batch_size=128, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print(f"Test set size: {len(clean_test_set)}")


In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        outputs = model(images.to(DEVICE))
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


y_true, y_pred = collect_predictions(model, clean_test_loader)
clean_acc = (y_true == y_pred).mean()
print(f"Clean test accuracy: {clean_acc:.4f}")


## Which classes get confused with each other

Ranks off-diagonal confusion-matrix cells by count, so the pairs that account
for the most mistakes in absolute terms surface first. Rate is relative to
how often that true class appears, so a rare class with a handful of mistakes
doesn't get lost next to a common class's larger raw counts.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
true_counts = cm.sum(axis=1)

pairs = []
for true_id in range(NUM_CLASSES):
    for pred_id in range(NUM_CLASSES):
        if true_id == pred_id:
            continue
        count = cm[true_id, pred_id]
        if count == 0:
            continue
        pairs.append({
            "true_class": true_id,
            "true_name": CLASS_NAMES[true_id],
            "pred_class": pred_id,
            "pred_name": CLASS_NAMES[pred_id],
            "count": int(count),
            "rate_of_true_class": count / true_counts[true_id],
        })

confusion_pairs = pd.DataFrame(pairs).sort_values("count", ascending=False).reset_index(drop=True)
confusion_pairs.to_csv(OUTPUT_DIR / "confusion_pairs.csv", index=False)
confusion_pairs.head(20)


In [ ]:
top_pairs = confusion_pairs.head(15)
labels = [f"{r.true_name}\n-> {r.pred_name}" for r in top_pairs.itertuples()]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(labels[::-1], top_pairs["count"][::-1])
ax.set_xlabel("Misclassified count (test set)")
ax.set_title("Top 15 confused class pairs")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "top_confused_pairs.png", dpi=150)
plt.show()


### A closer look at the top confusion

Sample actual misclassified images for the single most-confused pair, so the
mix-up can be sanity-checked visually (e.g. near-duplicate speed-limit digits,
similar triangular warning icons, etc.).

In [ ]:
top = confusion_pairs.iloc[0]
true_id, pred_id = int(top["true_class"]), int(top["pred_class"])

match_idx = np.where((y_true == true_id) & (y_pred == pred_id))[0][:8]

fig, axes = plt.subplots(1, len(match_idx), figsize=(2 * len(match_idx), 2.5))
for ax, idx in zip(axes, match_idx):
    img, _ = clean_test_set[idx]
    img = img * torch.tensor(IMAGENET_STD).view(3, 1, 1) + torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.axis("off")
fig.suptitle(f"True: {CLASS_NAMES[true_id]}  |  Predicted: {CLASS_NAMES[pred_id]}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "top_confusion_examples.png", dpi=150)
plt.show()


## Accuracy degradation under simulated conditions

For each of glare / motion blur / low light, sweeps severity from 0 (clean)
to 1 (extreme) and re-evaluates. Uses a fixed random subset of the test set
(same indices across all runs) so the comparison isn't muddied by which
images happened to be sampled, and so the full sweep finishes in a
reasonable time on CPU.

In [ ]:
SUBSET_SIZE = min(2000, len(clean_test_set))
SEVERITIES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

rng = np.random.default_rng(SEED)
subset_indices = rng.choice(len(clean_test_set), size=SUBSET_SIZE, replace=False)
print(f"Stress-testing on a fixed {SUBSET_SIZE}-image subset of the test set")


In [ ]:
def eval_condition(perturbation, severity):
    ds = GTSRB(
        root=str(DATA_DIR), split="test",
        transform=eval_transform(severity, perturbation), download=True,
    )
    loader = DataLoader(
        Subset(ds, subset_indices), batch_size=128, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    yt, yp = collect_predictions(model, loader)
    return (yt == yp).mean()


results = []
for name in PERTURBATIONS:
    for severity in SEVERITIES:
        acc = eval_condition(name, severity)
        results.append({"perturbation": name, "severity": severity, "accuracy": acc})
        print(f"{name:12s} severity={severity:.1f}  accuracy={acc:.4f}")

degradation = pd.DataFrame(results)
degradation.to_csv(OUTPUT_DIR / "degradation_results.csv", index=False)
degradation.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, group in degradation.groupby("perturbation"):
    ax.plot(group["severity"], group["accuracy"], marker="o", label=name)

ax.set_xlabel("Severity")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs. simulated condition severity")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "degradation_plot.png", dpi=150)
plt.show()


### Qualitative look at each condition

One sample image shown at each severity level, per condition, to sanity-check
that the simulated effect looks like what it claims to be before trusting the
accuracy numbers above.

In [ ]:
sample_idx = int(subset_indices[0])
raw_img, _ = GTSRB(root=str(DATA_DIR), split="test")[sample_idx]
raw_img = raw_img.resize((IMG_SIZE, IMG_SIZE))

fig, axes = plt.subplots(len(PERTURBATIONS), len(SEVERITIES), figsize=(2 * len(SEVERITIES), 2 * len(PERTURBATIONS)))
for row, name in enumerate(PERTURBATIONS):
    for col, severity in enumerate(SEVERITIES):
        transform = PERTURBATIONS[name](severity)
        img = transform(raw_img) if severity > 0 else raw_img
        ax = axes[row, col]
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f"severity={severity:.1f}")
        if col == 0:
            ax.set_ylabel(name)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "perturbation_examples.png", dpi=150)
plt.show()


## Summary

- `outputs/confusion_pairs.csv` / `top_confused_pairs.png` — which sign
  classes the model mixes up most, ranked by absolute count.
- `outputs/top_confusion_examples.png` — sample images for the single worst
  confusion pair.
- `outputs/degradation_results.csv` / `degradation_plot.png` — accuracy vs.
  severity for glare, motion blur, and low light.
- `outputs/perturbation_examples.png` — a visual sanity check of what each
  simulated condition actually looks like.

Re-run this notebook after any retrain to track whether these failure modes
improve.